# Submissão 1 — Modelo B (DNN em PyTorch)

In [ ]:
import sys
sys.path.insert(0,'../src')

import numpy as np
import pandas as pd
import pickle
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from src.text_features import CombinedVectorizer
from src.ffnn import FFNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

ID2LABEL  = {0: 'Google', 1: 'Anthropic', 2: 'Meta', 3: 'OpenAI', 4: 'Human'}
N_CLASSES = 5

In [ ]:
df_subm = pd.read_csv('subm1.csv', sep=';')

with open('../vectorizers/vectorizer_dnn.pkl', 'rb') as f:
    vec = pickle.load(f)

X_subm       = vec.transform(df_subm['Text'].tolist())
X_subm_dense = X_subm.toarray() if not isinstance(X_subm, np.ndarray) else X_subm

print(f'dados transformados: {X_subm_dense.shape}')

In [ ]:
input_dim = X_subm_dense.shape[1]
model_dnn = FFNN(input_dim=input_dim, n_classes=N_CLASSES,
                 topology=[128, 64], dropout=0.3).to(device)
model_dnn.load_state_dict(torch.load('../models/model_dnn.pt', map_location=device))
model_dnn.eval()

Xt     = torch.tensor(X_subm_dense, dtype=torch.float32)
loader = DataLoader(TensorDataset(Xt), batch_size=64)

preds_all = []
with torch.no_grad():
    for (xb,) in loader:
        logits = model_dnn(xb.to(device))
        preds_all.extend(logits.argmax(dim=1).cpu().tolist())

In [ ]:
df_subm['Label'] = [ID2LABEL[p] for p in preds_all]
print(df_subm['Label'].value_counts())

filename = 'subm1-g2-MEI-B.csv'
df_subm[['ID', 'Label']].to_csv(filename, index=False, sep=';')
print(f"ficheiro '{filename}' guardado.")